# S10 – Unidad 1. Evidencia de Aprendizaje 1
## Carga y creación de dataset

**Integrantes:** Daniela Cataño Hernández · John Freddy Ocampo Ochoa · Óscar Alonso De La Pava Suárez

**Flujo:** `HFDatasetLoader` → DataFrame → transformaciones → `DatasetSummarizer` → `resumen_dataset.txt`

Las clases contienen la lógica de carga y de resumen; este notebook coordina el flujo y realiza las transformaciones.

## 1. Importar librerías y clases

In [ ]:
import pandas as pd
from dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

print("pandas", pd.__version__)

## 2. Cargar el dataset desde Hugging Face
Se usa `scikit-learn/iris` (150 flores, 4 medidas y la especie). Requiere conexión a Internet.

In [ ]:
loader = HFDatasetLoader(dataset_id="scikit-learn/iris", split="train")
df_original = loader.load_data()

print("Dimensiones originales:", df_original.shape)
print("Columnas originales:", list(df_original.columns))
df_original.head()

## 3. Transformación 1 – Renombrar columnas
Se traducen los nombres al español y se usan minúsculas con guion bajo, para que sean más claros y fáciles de escribir.

In [ ]:
nombres_nuevos = {
    "Id": "id",
    "SepalLengthCm": "largo_sepalo_cm",
    "SepalWidthCm": "ancho_sepalo_cm",
    "PetalLengthCm": "largo_petalo_cm",
    "PetalWidthCm": "ancho_petalo_cm",
    "Species": "especie",
}
df_t1 = df_original.rename(columns=nombres_nuevos)

print("Antes:  ", list(df_original.columns))
print("Después:", list(df_t1.columns))
df_t1.head()

## 4. Transformación 2 – Seleccionar columnas
La columna `id` es solo un consecutivo y no aporta información sobre las flores, así que se conservan únicamente las medidas y la especie.

In [ ]:
columnas_trabajo = ["largo_sepalo_cm", "ancho_sepalo_cm",
                    "largo_petalo_cm", "ancho_petalo_cm", "especie"]
df_t2 = df_t1[columnas_trabajo].copy()

print("Columnas antes:", df_t1.shape[1], "→ después:", df_t2.shape[1])
df_t2.head()

## 5. Transformación 3 – Limpiar texto y crear una columna calculada
Se elimina el prefijo `Iris-` de la especie (por ejemplo `Iris-setosa` → `setosa`) y se crea `area_petalo_cm2` = largo × ancho del pétalo.

In [ ]:
df_t3 = df_t2.copy()
df_t3["especie"] = df_t3["especie"].str.replace("Iris-", "", regex=False)
df_t3["area_petalo_cm2"] = (df_t3["largo_petalo_cm"] * df_t3["ancho_petalo_cm"]).round(2)

print("Especies antes:  ", sorted(df_t2["especie"].unique()))
print("Especies después:", sorted(df_t3["especie"].unique()))
df_t3.head()

## 6. Transformación 4 – Filtrar filas
Se conservan solo las flores con sépalo de al menos 5 cm de largo (condición sencilla sobre una columna).

In [ ]:
umbral = 5.0
df_final = df_t3[df_t3["largo_sepalo_cm"] >= umbral].reset_index(drop=True)

print(f"Filas antes del filtro: {len(df_t3)}")
print(f"Filas después del filtro (largo_sepalo_cm >= {umbral}): {len(df_final)}")
print(df_final["especie"].value_counts())
df_final.head()

## 7. Generar el resumen TXT del DataFrame transformado
Se pasa `df_final` (no `df_original`) a `DatasetSummarizer`, de modo que el archivo describe los datos transformados.

In [ ]:
resumidor = DatasetSummarizer(df_final)
archivo_resumen = resumidor.generate_summary("resumen_dataset.txt")

with open(archivo_resumen, encoding="utf-8") as archivo:
    print(archivo.read())

## 8. Comprobación final

In [ ]:
print(f"DataFrame original: {df_original.shape[0]} filas × {df_original.shape[1]} columnas")
print(f"DataFrame final:    {df_final.shape[0]} filas × {df_final.shape[1]} columnas")
print("Resumen guardado en:", archivo_resumen)

## Explicación de las transformaciones

| # | Transformación | Operación | Efecto |
|---|---|---|---|
| 1 | Renombrar columnas | `rename` | Nombres en español y en minúsculas |
| 2 | Seleccionar columnas | `df[[...]]` | Se elimina `id` (6 → 5 columnas) |
| 3 | Limpiar texto y crear columna | `str.replace` + operación aritmética | Especie sin prefijo `Iris-`; nueva columna `area_petalo_cm2` (5 → 6 columnas) |
| 4 | Filtrar filas | máscara booleana | Solo flores con sépalo ≥ 5 cm (150 → menos filas) |

El archivo `resumen_dataset.txt` corresponde al DataFrame final, después de las cuatro transformaciones.